# Hand pose from two 3Di cameras: training baseline (Kaggle)

Input: the 4 IR images of one frame group (cam_A left/right, cam_B left/right). Target: the MANO fit for that frame
(`<session>/mano/smoothed.npz` from `fit_mano.py`: orient 3 + pose 45 + transl 3). Loss: parameters + 3D joints through
the MANO layer. Metric: joint error in mm (MPJPE) on the last 20 % of each session.

**The model here is a placeholder baseline (ResNet-18).** Swap the `make()` cell for the real model once it is chosen.

### Before running
1. Settings → Accelerator **GPU T4 x2**, **Internet on** (needs a phone-verified Kaggle account).
2. Add-ons → Secrets: `MANO_USER`, `MANO_PASS` = your login at mano.is.tue.mpg.de (licence accepted). Attach both to
   this notebook. MANO is downloaded from the official site each run and never stored in the output.
3. Set `DATA_URL` below: a Drive link (“anyone with the link”) to a **.zip of the hand sessions** (best) or to the
   dataset folder. Or attach the data as a Kaggle Dataset instead: if any `groups.json` is found under `/kaggle/input`,
   nothing is downloaded.
4. Run all (or Save Version → Save & Run All to run it in the background; `best.pt` lands in the output).

### How long
Rough, for the 7 hand sessions (~2,700 labelled frames, ~5–6 GB): download 3–15 min (depends on Drive), MANO < 1 min,
loading the images 1–3 min, training 30 epochs ~5–15 min. **The measured estimate for training is printed by the
“Time estimate” cell before training starts**, and the training cell prints the time left after each epoch.

In [ ]:
DATA_URL = ""     # Drive link: a .zip of the sessions (best) or the dataset folder
EPOCHS = 30
IMG = 256         # images are resized from 384 x 384 to IMG x IMG
BATCH = 64
LR = 3e-4
VAL_FRAC = 0.2    # the last 20 % (in time) of every session is validation

In [ ]:
import glob, io, json, math, os, shutil, subprocess, sys, time, zipfile
T_START = time.time()
took = lambda t: f"{(time.time() - t) / 60:.1f} min"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "smplx"], check=True)
REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/amitabh-anandpd/multi-3di-synced-ir-capture", REPO], check=True)
sys.path.insert(0, REPO)
print("setup", took(T_START))

## Dataset

In [ ]:
import gdown

def label_file(s):
    for n in ("smoothed.npz", "params.npz"):
        if os.path.exists(f"{s}/mano/{n}"):
            return f"{s}/mano/{n}"

t = time.time()
found = glob.glob("/kaggle/input/**/groups.json", recursive=True)
if found:
    print("using the attached Kaggle input, nothing downloaded")
else:
    assert DATA_URL, "set DATA_URL in the first code cell"
    DATA = "/tmp/data"  # /kaggle/working is capped at 20 GB
    os.makedirs(DATA, exist_ok=True)
    if "/folders/" in DATA_URL:
        gdown.download_folder(DATA_URL, output=DATA, quiet=False, remaining_ok=True)
    else:
        z = gdown.download(DATA_URL, output=f"{DATA}/data.zip", fuzzy=True, quiet=False)
        shutil.unpack_archive(z, DATA)
        os.remove(z)
    found = glob.glob(f"{DATA}/**/groups.json", recursive=True)
sessions = sorted({os.path.dirname(p) for p in found})
labelled = [s for s in sessions if label_file(s)]
for s in sessions:
    print(f"{'labelled  ' if s in labelled else 'no labels '} {s}")
assert labelled, "no session has mano/smoothed.npz or params.npz (fit_mano.py output, see HANDOFF.md)"
print(f"dataset {took(t)}, disk free {shutil.disk_usage('/tmp').free / 1e9:.0f} GB")

## MANO (official download, your login)

In [ ]:
import requests
MANO = f"{REPO}/mano"
os.makedirs(MANO, exist_ok=True)
t = time.time()
local = glob.glob("/kaggle/input/**/MANO_LEFT.pkl", recursive=True)
if local:
    for side in ("LEFT", "RIGHT"):
        shutil.copy(os.path.join(os.path.dirname(local[0]), f"MANO_{side}.pkl"), MANO)
else:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    r = requests.post("https://download.is.tue.mpg.de/download.php?domain=mano&resume=1&sfile=mano_v1_2.zip",
                      data={"username": sec.get_secret("MANO_USER"), "password": sec.get_secret("MANO_PASS")}, timeout=900)
    assert r.content[:2] == b"PK", f"MANO download failed (login / licence?): {r.text[:300]}"
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        for side in ("LEFT", "RIGHT"):
            name = next(n for n in z.namelist() if n.endswith(f"models/MANO_{side}.pkl"))
            open(f"{MANO}/MANO_{side}.pkl", "wb").write(z.read(name))
subprocess.run([sys.executable, f"{REPO}/tools/convert_mano.py"], check=True, capture_output=True)
print("MANO ready", took(t))

## Load frames + labels into memory

In [ ]:
import cv2
import numpy as np
from session_io import load_rows, load_session, read_pair

t = time.time()
X, Y, J, S = [], [], [], []  # images (4, IMG, IMG), params (51), joints (21, 3) mm, session index
for si, s in enumerate(labelled):
    z = np.load(label_file(s))
    size = load_session(s)["image_size"]
    a2b = {g["frames"]["cam_A"]: g["frames"]["cam_B"] for g in json.load(open(f"{s}/groups.json"))["groups"]}
    rows = {c: {int(r["frame_id"]): r for r in load_rows(f"{s}/{c}")} for c in ("cam_A", "cam_B")}
    n0 = len(X)
    for i, fa in enumerate(z["frames"].astype(int)):
        fb = a2b.get(fa)
        if fb is None or fa not in rows["cam_A"] or fb not in rows["cam_B"]:
            continue
        imgs = [*read_pair(f"{s}/cam_A", rows["cam_A"][fa], size), *read_pair(f"{s}/cam_B", rows["cam_B"][fb], size)]
        X.append(np.stack([cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA) for im in imgs]))
        Y.append(np.r_[z["orient"][i], z["pose"][i], z["transl"][i]])
        J.append(z["joints"][i])
        S.append(si)
    print(f"{os.path.basename(s)}: {len(X) - n0} of {len(z['frames'])} labelled frames")
X, Y, J, S = np.stack(X), np.stack(Y).astype(np.float32), np.stack(J).astype(np.float32), np.array(S)
BETAS = np.load(label_file(labelled[0]))["betas"].astype(np.float32)

tr, va = [], []  # time split per session: consecutive frames are near-duplicates, so no random split
for si in range(len(labelled)):
    idx = np.where(S == si)[0]
    cut = int(len(idx) * (1 - VAL_FRAC))
    tr += list(idx[:cut]); va += list(idx[cut:])
tr, va = np.array(tr), np.array(va)
print(f"{len(X)} frames ({X.nbytes / 1e9:.2f} GB), train {len(tr)}, val {len(va)}; {took(t)}")

## Model (placeholder: swap `make()` for the real one)

In [ ]:
import torch, torch.nn as nn, torchvision
from fit_mano import Hand

dev = "cuda"
hand = Hand(json.load(open(f"{REPO}/mano_shape.json"))["side"]); hand.m.to(dev)
betas = torch.tensor(BETAS, device=dev)
Xg, Yg, Jg = (torch.from_numpy(a).to(dev) for a in (X, Y, J))  # whole set fits on one T4
mu, sd = Yg[tr].mean(0), Yg[tr].std(0) + 1e-6

def joints(p):  # (n, 51) MANO params -> (n, 21, 3) mm, cam_A unit frame
    return hand(betas.expand(len(p), -1), p[:, :3], p[:, 3:48], p[:, 48:])[1]

def prep(x, train=False):
    x = x.float() / 255
    if train:
        x = x * torch.empty(len(x), 1, 1, 1, device=dev).uniform_(0.8, 1.2)  # brightness jitter
    return (x - 0.3) / 0.25

def make():
    net = torchvision.models.resnet18(weights="DEFAULT")
    w = net.conv1.weight.data.mean(1, keepdim=True).repeat(1, 4, 1, 1)
    net.conv1 = nn.Conv2d(4, 64, 7, 2, 3, bias=False); net.conv1.weight.data = w
    net.fc = nn.Linear(512, 51)
    net = net.to(dev)
    if torch.cuda.device_count() > 1:
        net = nn.DataParallel(net)
    opt = torch.optim.AdamW(net.parameters(), LR, weight_decay=1e-4)
    return net, opt, torch.amp.GradScaler("cuda")

def step(net, opt, scaler, i):
    with torch.autocast("cuda", torch.float16):
        out = net(prep(Xg[i], train=True))
    p = out.float() * sd + mu
    loss = nn.functional.mse_loss(out.float(), (Yg[i] - mu) / sd) + (joints(p) - Jg[i]).norm(dim=-1).mean() / 10
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return loss.item()

@torch.no_grad()
def evaluate(net, idx):  # per-frame joint error (mm), absolute and wrist-relative
    net.eval()
    Jp = torch.cat([joints(net(prep(Xg[idx[b:b + 256]])).float() * sd + mu) for b in range(0, len(idx), 256)])
    net.train()
    Jt = Jg[idx]
    rel = lambda a: a - a[:, :1]
    return (Jp - Jt).norm(dim=-1).mean(1).cpu().numpy(), (rel(Jp) - rel(Jt)).norm(dim=-1).mean(1).cpu().numpy()

with torch.no_grad():  # floor to beat: always predict the mean training pose
    e = (joints(mu[None]) - Jg[va]).norm(dim=-1).mean().item()
print(f"predict-the-mean baseline on val: {e:.1f} mm")

## Time estimate

In [ ]:
net, opt, scaler = make()
spe = math.ceil(len(tr) / BATCH)
for k in range(25):
    if k == 5:
        torch.cuda.synchronize(); t = time.time()
    step(net, opt, scaler, torch.from_numpy(np.random.choice(tr, BATCH)).to(dev))
torch.cuda.synchronize()
per_step = (time.time() - t) / 20
t = time.time(); evaluate(net, va); per_eval = time.time() - t
est = EPOCHS * (spe * per_step + per_eval)
print(f"{torch.cuda.device_count()} GPU(s), {per_step * 1000:.0f} ms/step, {spe} steps/epoch, eval {per_eval:.1f} s")
print(f"ESTIMATED TRAINING TIME: {est / 60:.1f} min for {EPOCHS} epochs (elapsed so far {took(T_START)})")
net, opt, scaler = make()  # start training from fresh weights

## Train

In [ ]:
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS * spe)
best, t = 1e9, time.time()
for ep in range(EPOCHS):
    perm = torch.from_numpy(np.random.permutation(tr)).to(dev)
    losses = []
    for b in range(spe):
        losses.append(step(net, opt, scaler, perm[b * BATCH:(b + 1) * BATCH])); sched.step()
    e, er = evaluate(net, va)
    if e.mean() < best:
        best = e.mean()
        torch.save({"model": (net.module if hasattr(net, "module") else net).state_dict(), "mu": mu.cpu(), "sd": sd.cpu(),
                    "img": IMG, "sessions": [os.path.basename(s) for s in labelled]}, "/kaggle/working/best.pt")
    left = (time.time() - t) / (ep + 1) * (EPOCHS - ep - 1)
    print(f"epoch {ep + 1:3d}/{EPOCHS}  loss {np.mean(losses):.3f}  val joints {e.mean():6.1f} mm  "
          f"wrist-relative {er.mean():5.1f} mm  {'*' if e.mean() == best else ' '}  ~{left / 60:.1f} min left")
print(f"training {took(t)}, best val {best:.1f} mm, total run {took(T_START)}")

## Result per session (best checkpoint)

In [ ]:
(net.module if hasattr(net, "module") else net).load_state_dict(torch.load("/kaggle/working/best.pt")["model"])
e, er = evaluate(net, va)
print(f"{'session':<22}{'val frames':>11}{'joints mm':>11}{'wrist-rel mm':>14}")
for si, s in enumerate(labelled):
    m = S[va] == si
    print(f"{os.path.basename(s):<22}{m.sum():>11}{e[m].mean():>11.1f}{er[m].mean():>14.1f}")
print(f"{'all':<22}{len(va):>11}{e.mean():>11.1f}{er.mean():>14.1f}")